# Unsway — Phase 6B baseline and multicouche extraction

This notebook evaluates all three prompt conditions on train/validation, but only the initial prompt on the fresh test split. It then extracts pressured final-token activations from train/validation across all 12 GPT-2 small residual layers.

## Update the repository and install dependencies

In [ ]:
from pathlib import Path

repo = Path("/content/Unsway")
if (repo / ".git").is_dir():
    !git -C /content/Unsway pull --ff-only
else:
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!pip install -q -e '.[dev]'

## Verify the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print(torch.cuda.get_device_name(0))

## Rebuild the checksum-pinned datasets

In [ ]:
!python -m unsway.cli.phase1 --config configs/phase1.yaml
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage data

## Run the leakage-safe baseline

This is the only Phase 6B step that touches test examples, and it scores their initial prompts only.

In [ ]:
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage baseline

## Check eligibility before extraction

In [ ]:
import json

baseline = json.loads(Path("reports/phase6_baseline.json").read_text())
test_initial = baseline["test_initial_only"]
print(json.dumps(test_initial["metrics"], indent=2))
assert test_initial["pressure_scored"] is False
assert test_initial["control_scored"] is False
assert baseline["status"] == "ready_for_frozen_test", baseline["status"]
print("Eligibility guardrail passed; the pressure/control test remains unopened.")

## Extract 12-layer train/validation activations

In [ ]:
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage extract

In [ ]:
extraction = json.loads(Path("reports/phase6_extraction.json").read_text())
print(
    json.dumps(
        {
            "shape": extraction["shape"],
            "split_counts": extraction["split_counts"],
            "behavior_counts": extraction["behavior_counts"],
            "test_examples_extracted": extraction["test_examples_extracted"],
        },
        indent=2,
    )
)
assert extraction["test_examples_extracted"] == 0

## Back up the expensive artifacts to Google Drive

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/Unsway/phase6b")
backup.mkdir(parents=True, exist_ok=True)
for source in [
    Path("data/processed/phase6_train_validation_predictions.jsonl"),
    Path("data/processed/phase6_test_initial_predictions.jsonl"),
    Path("data/processed/phase6/multilayer_final_activations.safetensors"),
    Path("data/processed/phase6/multilayer_examples.json"),
    Path("reports/phase6_baseline.json"),
    Path("reports/phase6_extraction.json"),
]:
    shutil.copy2(source, backup / source.name)
print("Backed up to", backup)

## Download the two small reports for Git

In [ ]:
from google.colab import files

files.download("reports/phase6_baseline.json")
files.download("reports/phase6_extraction.json")